# Music Annotation Project A5
## Model Generation: A1-Based Selective Safe Annotation Cleanup

**Runtime: GPU.** Select a T4 GPU and run all cells.

This notebook creates the A5 development model before any A5 test notebook is run.

**Contract**
- Eligible population: pixels in the frozen `AMBIGUOUS` route that A1 preserves.
- A5 default: `KEEP_A1`, copying the observed byte exactly.
- Learned positive class: `SAFE_ANNOTATION_CLEANUP`.
- Truth-uncertain pixels are excluded from supervised loss and remain `KEEP_A1`.
- Clean pages create development labels only. Clean pixels and truth-derived maps never enter model inference.
- Models are trained out of fold by source group and every checkpoint, calibrator, schema and fold map is persisted.
- This notebook generates a development model bundle. It does not authorise production use.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn pyarrow torch torchvision joblib matplotlib


## 1. Configuration and run identity


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, random, sys, platform, gc, shutil, time
import cv2, torch, joblib
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
MANIFEST_PATH=ROOT/'development_score_manifest.csv'
OUTPUT_ROOT=ROOT/'a5_safe_annotation_cleanup_model'
SEED=42
LOCAL_PATCH=33
WIDE_PATCH=97
DARK_THRESHOLD=200
MAX_PER_PAGE_CLASS=25000
BATCH=256
EPOCHS=15
LR=1e-3
WEIGHT_DECAY=1e-4
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge','paper_support']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('A5 model generation requires a GPU runtime.')
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_a5_model_generation'
OUT=OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; MODELS=OUT/'models'; CALIBRATORS=OUT/'calibrators'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; REPLAY=OUT/'replay'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,MODELS,CALIBRATORS,TABLES,AUDIT,REPLAY,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('GPU:',torch.cuda.get_device_name(0)); print('Output:',OUT)


## 2. Resolve paired development data and frozen V7.5 arrays


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def latest(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
def image_map(folder):
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def object_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

if not MANIFEST_PATH.exists(): raise RuntimeError(f'Missing {MANIFEST_PATH}')
manifest=pd.read_csv(MANIFEST_PATH,dtype=str)
missing={'page','source_group'}-set(manifest.columns)
if missing: raise RuntimeError(f'Manifest missing columns: {sorted(missing)}')
if manifest.page.duplicated().any(): raise RuntimeError('Duplicate page values in development manifest.')
if manifest.source_group.isna().any() or (manifest.source_group.str.strip()=='').any(): raise RuntimeError('Missing source_group values.')
shutil.copy2(MANIFEST_PATH,INPUTS/'development_score_manifest.csv')
V75=latest(V75_ROOT,['tables/cache_status.csv'])
if V75 is None: raise RuntimeError('Complete V7.5 development run not found.')
status=pd.read_csv(V75/'tables'/'cache_status.csv')
cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
clean_map=image_map(ROOT/'source'/'clean'); observed_map=image_map(ROOT/'aligned')
pages=sorted(set(manifest.page.astype(str)) & set(cache) & set(clean_map) & set(observed_map))
if set(pages)!=set(manifest.page.astype(str)):
    raise RuntimeError(f'Missing paired inputs: {sorted(set(manifest.page.astype(str))-set(pages))}')
meta=manifest.set_index('page')
print('V7.5:',V75); print('Pages:',len(pages),'Source groups:',manifest.source_group.nunique())


## 3. Build A5 labels and observable channels


In [ ]:
def printed_ink(clean):
    return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def truth_maps(clean):
    ink=printed_ink(clean); k=np.ones((3,3),np.uint8)
    near=cv2.dilate(ink.astype(np.uint8),k)>0
    eroded=cv2.erode(ink.astype(np.uint8),k)>0
    unstable=np.logical_xor(near,eroded)
    return near,unstable
def background_surface(obs,mask,sigma=45.0):
    visible=(~mask)&(obs>=200)
    num=cv2.GaussianBlur(obs.astype(np.float32)*visible,(0,0),sigma)
    den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),sigma)
    fallback=cv2.GaussianBlur(obs,(0,0),sigma)
    return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

runtime={}; audit=[]
required=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
for page in pages:
    with np.load(cache[page]) as z:
        absent=[k for k in required if k not in z.files]
        if absent: raise RuntimeError(f'{page}: missing V7.5 arrays {absent}')
        a={k:z[k] for k in required}
    clean=cv2.imread(str(clean_map[page]),0); obs=cv2.imread(str(observed_map[page]),0)
    if clean is None or obs is None or clean.shape!=obs.shape: raise RuntimeError(f'{page}: image mismatch')
    mask=a['mask']>0; ambiguous=(a['g80_ambiguous']>0)&mask
    near,unstable=truth_maps(clean)
    eligible=ambiguous&(obs<DARK_THRESHOLD)
    label=np.full(obs.shape,-1,np.int8)
    label[eligible&~near&~unstable]=1  # SAFE_ANNOTATION_CLEANUP
    label[eligible&near&~unstable]=0   # KEEP_A1: genuine print
    label[eligible&unstable]=2         # truth uncertain
    paper=background_surface(obs,mask)
    norm=cv2.divide(obs,np.maximum(paper,1),scale=255)
    paper_support=cv2.GaussianBlur(((~mask)&(obs>=220)).astype(np.float32),(0,0),9)
    runtime[page]={
        'obs':obs,'clean':clean,'mask':mask,'ambiguous':ambiguous,'a1':obs.copy(),'cleanup':paper,
        'label':label,'norm':norm,'depth':cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5),
        'horizontal':a['g80_horizontal']>0,'vertical':a['g80_vertical']>0,'compact':a['g80_compact']>0,
        'preserve':a['g80_preserve']>0,'remove':a['g80_remove']>0,'edge':cv2.Canny(obs,70,170)>0,
        'paper_support':paper_support
    }
    audit.append({'page':page,'source_group':meta.loc[page,'source_group'],'eligible':int(eligible.sum()),
                  'safe_cleanup':int((label==1).sum()),'keep_a1_print':int((label==0).sum()),
                  'truth_uncertain':int((label==2).sum()),'cache_sha256':sha256_file(cache[page])})
pd.DataFrame(audit).to_csv(AUDIT/'input_and_label_audit.csv',index=False)
print(pd.DataFrame(audit)[['eligible','safe_cleanup','keep_a1_print','truth_uncertain']].sum())


## 4. Deterministic source-group folds and sampled training roster


In [ ]:
rng=np.random.default_rng(SEED); rows=[]
for page,z in runtime.items():
    group=str(meta.loc[page,'source_group'])
    for cls in [0,1]:
        ys,xs=np.where(z['label']==cls); idx=np.arange(len(ys))
        take=idx if len(idx)<=MAX_PER_PAGE_CLASS else rng.choice(idx,MAX_PER_PAGE_CLASS,replace=False)
        rows.extend({'page':page,'source_group':group,'y':int(ys[j]),'x':int(xs[j]),'truth':int(cls)} for j in take)
samples=pd.DataFrame(rows).sort_values(['source_group','page','y','x']).reset_index(drop=True)
if samples.empty or samples.truth.nunique()!=2: raise RuntimeError('A5 requires both supervised classes.')
groups=samples.source_group
splitter=LeaveOneGroupOut() if groups.nunique()<=10 else GroupKFold(n_splits=min(10,groups.nunique()))
fold=np.full(len(samples),-1,int); mapping={}; split_rows=[]
for f,(tr,te) in enumerate(splitter.split(samples,samples.truth,groups),1):
    tg=set(groups.iloc[tr]); eg=set(groups.iloc[te]); overlap=tg&eg
    if overlap: raise RuntimeError(f'Fold {f}: source leakage {overlap}')
    fold[te]=f
    for g in eg: mapping[g]=f
    split_rows.append({'fold':f,'train_rows':len(tr),'test_rows':len(te),'test_pages':samples.iloc[te].page.nunique(),'test_groups':'|'.join(sorted(eg))})
samples['fold']=fold; samples['sample_id']=np.arange(len(samples))
samples.to_parquet(TABLES/'a5_training_sample_manifest.parquet',index=False)
pd.DataFrame(split_rows).to_csv(AUDIT/'source_group_split_audit.csv',index=False)
pd.DataFrame([{'source_group':g,'fold':f} for g,f in sorted(mapping.items())]).to_csv(INPUTS/'fold_map.csv',index=False)
print(samples.groupby(['fold','truth']).size())


## 5. Dual-context A5 CNN


In [ ]:
LOCAL_HALF=LOCAL_PATCH//2; WIDE_HALF=WIDE_PATCH//2
padded={}
for page,z in runtime.items():
    padded[page]={}
    for k in CHANNELS:
        arr=z[k].astype(np.float32)
        if k in ['obs','norm']: arr/=255.0
        padded[page][k]=np.pad(arr,WIDE_HALF,mode='reflect')
page_list=pages; page_index={p:i for i,p in enumerate(page_list)}
P=samples.page.map(page_index).to_numpy(np.int16); Y=samples.y.to_numpy(np.int32); X=samples.x.to_numpy(np.int32); T=samples.truth.to_numpy(np.int8)
def patch(pid,y,x,size):
    page=page_list[int(pid)]; half=size//2; offset=WIDE_HALF-half
    return np.stack([padded[page][k][y+offset:y+offset+size,x+offset:x+offset+size] for k in CHANNELS]).astype(np.float32)
class A5Dataset(Dataset):
    def __init__(self,idx): self.idx=np.asarray(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        j=int(self.idx[i])
        return torch.from_numpy(patch(P[j],Y[j],X[j],LOCAL_PATCH)),torch.from_numpy(patch(P[j],Y[j],X[j],WIDE_PATCH)),torch.tensor(int(T[j]),dtype=torch.long)
def loader(idx,shuffle,seed):
    g=torch.Generator(); g.manual_seed(seed)
    return DataLoader(A5Dataset(idx),batch_size=BATCH,shuffle=shuffle,num_workers=2,pin_memory=True,generator=g)
class Encoder(nn.Module):
    def __init__(self,n):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(n,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
    def forward(self,x): return self.net(x).flatten(1)
class A5DualContextCNN(nn.Module):
    def __init__(self,n=len(CHANNELS)):
        super().__init__(); self.local=Encoder(n); self.wide=Encoder(n); self.head=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Dropout(.15),nn.Linear(64,2))
    def forward(self,local,wide): return self.head(torch.cat([self.local(local),self.wide(wide)],1))
def fit_model(train_idx):
    reset_seed(); model=A5DualContextCNN().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    counts=np.bincount(T[train_idx],minlength=2); weights=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE)
    loss_fn=nn.CrossEntropyLoss(weight=weights); history=[]
    for ep in range(EPOCHS):
        model.train(); total=0
        for xl,xw,y in loader(train_idx,True,SEED+ep):
            xl=xl.to(DEVICE,non_blocking=True); xw=xw.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True); loss=loss_fn(model(xl,xw),y); loss.backward(); opt.step(); total+=loss.item()*len(y)
        history.append(total/len(train_idx)); print(f'epoch {ep+1:02d}/{EPOCHS}: {history[-1]:.6f}')
    return model,history
def predict(model,idx):
    out=[]; model.eval()
    with torch.no_grad():
        for xl,xw,_ in loader(idx,False,SEED): out.extend(torch.softmax(model(xl.to(DEVICE),xw.to(DEVICE)),1)[:,1].cpu().numpy())
    return np.asarray(out,np.float64)


## 6. Train and persist out-of-fold A5 models


In [ ]:
raw=np.full(len(samples),np.nan); model_rows=[]; history_rows=[]
for f in sorted(samples.fold.unique()):
    print('FOLD',f); tr=np.where(samples.fold.to_numpy()!=f)[0]; te=np.where(samples.fold.to_numpy()==f)[0]
    t0=time.time(); model,hist=fit_model(tr); raw[te]=predict(model,te)
    path=MODELS/f'a5_fold{f}_seed{SEED}.pt'
    torch.save({'state_dict':model.state_dict(),'channels':CHANNELS,'local_patch':LOCAL_PATCH,'wide_patch':WIDE_PATCH},path)
    model_rows.append({'fold':int(f),'checkpoint':str(path),'sha256':sha256_file(path),'train_rows':len(tr),'test_rows':len(te),'seconds':time.time()-t0})
    history_rows.extend({'fold':int(f),'epoch':i+1,'loss':v} for i,v in enumerate(hist))
    del model; gc.collect(); torch.cuda.empty_cache()
if not np.isfinite(raw).all(): raise RuntimeError('Incomplete OOF scores.')
np.save(TABLES/'a5_raw_oof.npy',raw)
pd.DataFrame(model_rows).to_csv(AUDIT/'model_inventory.csv',index=False)
pd.DataFrame(history_rows).to_csv(TABLES/'training_history.csv',index=False)


## 7. Cross-fitted isotonic calibration and OOF model card


In [ ]:
cal=np.full(len(samples),np.nan); cal_rows=[]
for f in sorted(samples.fold.unique()):
    tr=np.where(samples.fold.to_numpy()!=f)[0]; te=np.where(samples.fold.to_numpy()==f)[0]
    iso=IsotonicRegression(out_of_bounds='clip').fit(raw[tr],T[tr]); cal[te]=iso.predict(raw[te])
    path=CALIBRATORS/f'a5_fold{f}_isotonic.joblib'; joblib.dump(iso,path)
    replay=joblib.load(path).predict(raw[te])
    if not np.array_equal(replay,cal[te]): raise RuntimeError(f'Fold {f}: calibrator replay failed')
    cal_rows.append({'fold':int(f),'calibrator':str(path),'sha256':sha256_file(path),'exact_replay':True})
if not np.isfinite(cal).all(): raise RuntimeError('Incomplete calibrated OOF probabilities.')
oof=samples.copy(); oof['raw_safe_cleanup']=raw; oof['p_safe_cleanup']=cal
oof.to_parquet(TABLES/'a5_oof_predictions.parquet',index=False)
pd.DataFrame(cal_rows).to_csv(AUDIT/'calibrator_inventory.csv',index=False)
print(oof.groupby('truth').p_safe_cleanup.describe())


## 8. Fit the deployable development bundle on all permitted paired-development samples


In [ ]:
model,full_history=fit_model(np.arange(len(samples)))
model_path=MODELS/'a5_development_full_model.pt'
torch.save({'state_dict':model.state_dict(),'channels':CHANNELS,'local_patch':LOCAL_PATCH,'wide_patch':WIDE_PATCH,'architecture':'A5DualContextCNN'},model_path)
full_iso=IsotonicRegression(out_of_bounds='clip').fit(raw,T)
cal_path=CALIBRATORS/'a5_development_full_isotonic.joblib'; joblib.dump(full_iso,cal_path)
policy={
 'policy_status':'UNFROZEN_DEVELOPMENT_MODEL_REQUIRES_THRESHOLD_TEST',
 'positive_class':'SAFE_ANNOTATION_CLEANUP','fallback':'KEEP_A1',
 'eligible_population':'frozen AMBIGUOUS route and observed intensity < 200',
 'cleanup_treatment':'frozen local paper estimate from V7.5-compatible background surface',
 'threshold':None,
 'note':'Testing notebook must select and then freeze the operating threshold under the A1 non-inferiority and annotation-gain gates; no output is authorised before that step.'
}
(INPUTS/'channel_schema.json').write_text(json.dumps({'channels':CHANNELS,'local_patch':LOCAL_PATCH,'wide_patch':WIDE_PATCH,'dtype':'float32','padding':'reflect'},indent=2))
(OUT/'a5_policy_unfrozen.json').write_text(json.dumps(policy,indent=2))


## 9. Replay pack, manifest and final upload package


In [ ]:
replay_idx=np.linspace(0,len(samples)-1,min(256,len(samples)),dtype=int)
np.savez_compressed(REPLAY/'replay_samples.npz',sample_id=samples.iloc[replay_idx].sample_id.to_numpy(),raw=raw[replay_idx],calibrated=cal[replay_idx],truth=T[replay_idx])
identity={
 'created_utc':datetime.now(timezone.utc).isoformat(),'version':'A5-model-generation-v1','development_only':True,'promotion_authority':False,
 'v75_reference':str(V75),'manifest_sha256':sha256_file(INPUTS/'development_score_manifest.csv'),'fold_map_sha256':sha256_file(INPUTS/'fold_map.csv'),
 'model_sha256':sha256_file(model_path),'calibrator_sha256':sha256_file(cal_path),'schema_sha256':sha256_file(INPUTS/'channel_schema.json'),
 'pages':len(pages),'source_groups':int(manifest.source_group.nunique()),'samples':len(samples),'class_counts':samples.truth.value_counts().sort_index().to_dict(),
 'seed':SEED,'epochs':EPOCHS,'batch':BATCH,'lr':LR,'weight_decay':WEIGHT_DECAY,'models':model_rows,'calibrators':cal_rows,
 'threshold_frozen':False,'independent_data_used':False
}
(OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))
summary=[
 'A5 MODEL GENERATION COMPLETE','',f'Run: {OUT}',f'Pages: {len(pages)}',f'Source groups: {manifest.source_group.nunique()}',f'Supervised samples: {len(samples)}',
 f'Full model: {model_path.name}',f'Full calibrator: {cal_path.name}','',
 'This creates the A5 model bundle. The operating threshold is intentionally not frozen here.','Run the A5 evaluation notebook next to select and test the threshold against A1 under the pre-registered safety and annotation-cleanup gates.'
]
(OUT/'executive_summary.txt').write_text('
'.join(summary))
artefacts=[OUT/'executive_summary.txt',OUT/'experiment_identity.json',OUT/'a5_policy_unfrozen.json',model_path,cal_path,INPUTS/'channel_schema.json',INPUTS/'fold_map.csv',TABLES/'a5_oof_predictions.parquet',AUDIT/'model_inventory.csv',AUDIT/'calibrator_inventory.csv',AUDIT/'input_and_label_audit.csv']
inv=[]
for p in artefacts:
    dst=UPLOAD/p.name; shutil.copy2(p,dst); inv.append({'file':p.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False)
shutil.make_archive(str(OUT/'A5_model_generation_upload_bundle'),'zip',UPLOAD)
print('
'.join(summary)); print('Bundle:',OUT/'A5_model_generation_upload_bundle.zip')


## What to run next

Use the generated `a5_development_full_model.pt`, `a5_development_full_isotonic.joblib`, `channel_schema.json` and OOF predictions in the A5 evaluation notebook. The evaluator must freeze the cleanup threshold and numerical A1 non-inferiority/annotation-gain gates before treating the model as a testable A5 policy.
